# AII 600. Lab 5: how much does a sample tell you?

Due before class, 30 September 2026.

Upload this notebook on Canvas, with every code cell executed and the output left in. Credit is on-time submission. I do not mark the lab.

Python kernel. R is allowed: switch to an R kernel and rewrite the code cells. Ask the tutor to translate `check`.

Five questions from the estimation lecture, one per exercise:

| Ex. | Question | Tool |
|---|---|---|
| 1 | What share of calls beat the timeout? | read a CDF, $F(x)=P(X\le x)$ |
| 2 | Which timeout do 99% of calls beat? Can $F$ simulate calls? | invert $F$ |
| 3 | How much do 10 test results tell you, and how much do 640? | likelihood, on data you simulate |
| 4 | Why do averages look Normal, and where do they still not? | sampling distribution of $\bar X$ |
| 5 | Is 98.6°F a normal body temperature? *Optional, hard.* | Normal MLE |

Work each numeric question on paper first, then type the expression: the midterm has no computer and no test cell. Do not paste a number from the slides. After you finish, copy the formulas you actually used onto your cheat sheet.


## How this notebook works

1. Run the next cell once. It defines `check`.
2. Each exercise has an answer cell and a test cell. In the answer cell, replace every `None` and run it. Then run the test cell.
3. The test cell prints `ok  name` when that value is right to six decimals, `name is not right` when it is not (it never shows the target), and `replace None for name` when you left a `None`. Leave the output in the notebook.
4. Written cells have no test. Replace YOUR ANSWER HERE, then ask your tutor whether the reasoning holds. Prediction cells are for you: write them before you run the code, and do not go back and fix them.

When a check fails, the usual causes are: a number rounded too early (type `1 - norm.cdf(...)`, not `0.159`); the variance where scipy wants the standard deviation; a forgotten $1-F$ for an upper tail; $<$ where the question says $\le$ for a discrete variable; $n-1$ where the lab asks for $n$.

Quick reference. scipy's `scale` is the standard deviation for `norm` and the mean for `expon`. R's Exponential functions take the rate, $1/\theta$.

| You want | Python | R |
|---|---|---|
| $F(x)$ for $N(\mu,\sigma^2)$ | `norm.cdf(x, loc=mu, scale=sigma)` | `pnorm(x, mu, sigma)` |
| its quantile $F^{-1}(u)$ | `norm.ppf(u, loc=mu, scale=sigma)` | `qnorm(u, mu, sigma)` |
| Exponential with mean $\theta$ | `expon.cdf(x, scale=theta)`, `expon.ppf(u, scale=theta)` | `pexp(x, 1/theta)`, `qexp(u, 1/theta)` |
| Binomial$(n,p)$ | `binom.pmf(k, n, p)`, `rng.binomial(n, p)` | `dbinom(k, n, p)`, `rbinom(1, n, p)` |
| $m$ random draws | `rng.uniform(size=m)`, `rng.exponential(scale=theta, size=m)` | `runif(m)`, `rexp(m, 1/theta)` |

`numpy`, `matplotlib`, `scipy.stats`, and `pandas` are allowed. Ask the tutor for syntax; do the probability yourself.


In [ ]:
from hashlib import sha256

def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"replace None for {name}")
    s = f"{round(float(value), ndigits):.{ndigits}f}"
    got = sha256(f"aii600-lab5|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"{name} is not right"
    print("ok ", name)


## Exercise 1: read $F$ (warm-up)

$F(x)=P(X\le x)$ answers one question: what share is at or below $x$? Three settings, the same reading. Two identities do most of the work: $P(X>a)=1-F(a)$ and $P(a<X\le b)=F(b)-F(a)$.

**(a) Data.** Eight calls to a service took $0.4,\ 0.6,\ 0.9,\ 1.1,\ 1.5,\ 1.8,\ 2.4,\ 3.1$ seconds. The client gives up at 2 seconds. The empirical $F(2)$ is the share of these eight calls that finished in time: `f2_emp`.

**(b) A Normal model.** SAT scores are $N(1000, 200^2)$. On paper, standardise first: $z=(1200-1000)/200$.

- `p_above_1200` $=P(\mathrm{SAT}>1200)$ and `p_band` $=P(800<\mathrm{SAT}<1200)$.
- A scholarship goes to the top 10% of scores. Its cutoff is the $0.9$ quantile: `q90`.

**(c) A discrete law.** A fair die: `f2_die` $=F(2)$, `f4_die` $=F(4)$, `f_diff` $=F(4)-F(2)$, and `p_2to4` $=P(2\le X\le 4)$. Careful: the last two are not the same event.


In [ ]:
import numpy as np
from scipy.stats import norm

times = np.array([0.4, 0.6, 0.9, 1.1, 1.5, 1.8, 2.4, 3.1])   # seconds
f2_emp = None          # share of the eight calls done within 2 seconds
p_above_1200 = None    # P(SAT > 1200)
p_band = None          # P(800 < SAT < 1200)
q90 = None             # top-10% cutoff: the 0.9 quantile of SAT
f2_die = None          # F(2) for a fair die
f4_die = None          # F(4)
f_diff = None          # F(4) - F(2)
p_2to4 = None          # P(2 <= X <= 4)
f2_emp, p_above_1200, p_band, q90, f2_die, f4_die, f_diff, p_2to4


In [ ]:
check("f2_emp", f2_emp, "d7b6c7b10e673922")
check("p_above_1200", p_above_1200, "4c6c211314ad585f")
check("p_band", p_band, "57eb4a84372a0dce")
check("q90", q90, "0700dc205bcde43f")
check("f2_die", f2_die, "89f3628150051b72")
check("f4_die", f4_die, "6ec07cf3e91f8f72")
check("f_diff", f_diff, "d284912490ed21fc")
check("p_2to4", p_2to4, "5e5bc95652217bc7")
print("ok  exercise 1 cdf")


**Write.** (a) `f_diff` and `p_2to4` differ. Which die faces does each one count, and why does switching $<$ to $\le$ change the die answer but not `p_band`? Two sentences.

(b) The eight calls say $F(2)=0.75$. The lecture's Uniform$(0,4)$ model says $F(2)=0.5$. Your manager wants one number: the chance that the next call beats the timeout. Which do you report, and what evidence would change your mind? Three sentences.

YOUR ANSWER HERE


## Exercise 2: turn $F$ inside out

Your app calls a model API. Latency $X$, in seconds, is Exponential with mean $0.8$:
$$F(x)=1-e^{-x/0.8},\qquad x\ge 0.$$
$F$ answers "what share of calls has finished by time $x$?" Its inverse answers the reverse question, "by what time has a share $u$ finished?" That time is the quantile $F^{-1}(u)$.

1. **Invert $F$ on paper.** Set $u=F(x)$ and solve for $x$. Sanity check: the median, $F^{-1}(0.5)$, is about 0.55 seconds. It sits below the mean of 0.8 because a few slow calls pull the mean up.
2. **Pick a timeout.** `q99` is the $0.99$ quantile, a timeout that 99% of calls beat. Cross-check it with `expon.ppf(0.99, scale=0.8)`: a library's `ppf` is $F^{-1}$.
3. **Five draws by hand.** Push $u=0.1,\ 0.3,\ 0.5,\ 0.7,\ 0.9$ through your $F^{-1}$ and store the results, in order, as `x1` … `x5`.
4. **Ten thousand calls.** The starter makes 10,000 uniform numbers `u`. Push all of them through $F^{-1}$ at once (`np.log` works on a whole array) and store the result as `draws`. Store `frac_late`, the share of `draws` above `q99`, then uncomment `show_draws(draws, q99)`. Did your timeout do its job?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import expon

theta = 0.8     # mean latency, seconds


def show_draws(draws, timeout):
    xs = np.linspace(0, 6, 300)
    plt.figure()
    plt.hist(draws, bins=60, range=(0, 6), density=True,
             color="steelblue", edgecolor="white", label="your draws")
    plt.plot(xs, expon.pdf(xs, scale=theta), color="black", linewidth=2,
             label="Exponential density")
    plt.axvline(timeout, color="firebrick", linestyle="--", label="timeout q99")
    plt.xlabel("latency (seconds)")
    plt.legend()
    plt.show()


q99 = None
x1 = None
x2 = None
x3 = None
x4 = None
x5 = None

rng = np.random.default_rng(600)
u = rng.uniform(size=10_000)     # 10,000 random percentiles
draws = None                     # F^{-1}(u): one latency per u
frac_late = None                 # share of draws above q99
# show_draws(draws, q99)
q99, x1, x2, x3, x4, x5, frac_late


In [ ]:
check("q99", q99, "875a28eabab6a1b6")
check("x1", x1, "31846d7c214101ac")
check("x2", x2, "6096456c943f3b60")
check("x3", x3, "2e05f9a1deabb27e")
check("x4", x4, "e9049efc24631d07")
check("x5", x5, "806de3df38cb7b68")
print("ok  exercise 2 inverse cdf")
import numpy as np
assert draws is not None and frac_late is not None, "replace every None"
assert len(draws) == len(u), "one draw per u"
assert abs(np.mean(draws) - 0.8) < 0.04, "draws should average about 0.8 seconds"
assert abs(frac_late - 0.01) < 0.005, "about 1% of calls should outlast q99"
print("ok  exercise 2 simulation")


**Write.** Your five `x`'s look like a quantile table, yet the 10,000 `draws` behave like real latencies. Why is $X=F^{-1}(U)$, with $U\sim\mathrm{Uniform}(0,1)$, a draw from $F$? Start from $P\big(F^{-1}(U)\le x\big)$. Two or three sentences.

YOUR ANSWER HERE


## Exercise 3: watch the likelihood close in

In practice you never see a classifier's true accuracy. You see test results, and the likelihood works backwards from them. Here you play nature: the true accuracy is $p_{\mathrm{true}}=0.8$, you generate the test results yourself, and you watch how well the likelihood recovers $0.8$ as the test set grows.

Each test example is right with probability $p$, independently of the others, so the number right out of $n$ is $k\sim\mathrm{Binomial}(n,p)$. For an observed $k$,
$$L(p)\propto p^k(1-p)^{n-k},\qquad \ell(p)=k\log p+(n-k)\log(1-p).$$
The data stay fixed and $p$ moves. The lecture's eight heads in ten tosses is $k=8$, $n=10$.

1. **On paper.** Set $\ell'(p)=0$ and show that the maximiser is $\hat p=k/n$.
2. **Generate.** For each $n$ in `sizes` ($10, 40, 160, 640$), draw the number right with `rng.binomial(n, p_true)`. Store the four counts in `ks` and the four estimates $k/n$ in `p_hats`, in the order of `sizes`.
3. **Plot.** For each $n$, compute `loglik`, the value of $\ell(p)$ at every point of the grid `ps`, and plot the relative likelihood `np.exp(loglik - loglik.max())`. Every such curve peaks at height 1, so the four differ only in where they peak and how wide they are. Store where each one peaks, `ps[np.argmax(loglik)]`, in `grid_peaks`. Draw all four curves on one figure with a legend, and mark $p_{\mathrm{true}}$ with a vertical line.
4. **Perfect scores.** How often does this classifier ace a test? Store `p_perfect_10` $=P(k=10)$ for $n=10$ and `p_perfect_40` $=P(k=40)$ for $n=40$, both at $p=0.8$. On paper each is a single power; in code, `binom.pmf(k, n, p)`.


**Predict before you run.** One line each. A wrong prediction is useful; a skipped one is not.

1. Will the four curves peak at the same place? Exactly at 0.8?
2. Each test set is 4 times the size of the last. How much narrower is each curve than the one before?
3. If you rerun with a different seed, which curve moves the most?

YOUR PREDICTION HERE


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

rng = np.random.default_rng(10)       # change the seed to rerun the experiment
p_true = 0.8                          # you know this; the analyst does not
sizes = [10, 40, 160, 640]            # test-set sizes
ps = np.linspace(0.001, 0.999, 999)   # candidate values of p

ks = None             # number right at each size, in the order of sizes
p_hats = None         # k / n at each size
grid_peaks = None     # where each log-likelihood peaks on the grid ps
p_perfect_10 = None   # P(10 right out of 10) when p = 0.8
p_perfect_40 = None   # P(40 right out of 40) when p = 0.8

# for each n: draw k, compute loglik on ps, plot np.exp(loglik - loglik.max())
# then mark p_true with plt.axvline, add a legend, and plt.show()
ks, p_hats, grid_peaks, p_perfect_10, p_perfect_40


In [ ]:
import numpy as np
assert ks is not None and p_hats is not None and grid_peaks is not None, "replace every None"
n_ = np.array([10, 40, 160, 640])
k_ = np.asarray(ks, dtype=float)
assert k_.shape == (4,) and len(p_hats) == 4 and len(grid_peaks) == 4, "one value per test-set size"
assert np.all((k_ >= 0) & (k_ <= n_)) and np.allclose(k_, np.round(k_)), "each k is a whole number from 0 to n"
assert np.allclose(p_hats, k_ / n_), "each p_hat is k / n"
assert np.all(np.abs(np.asarray(grid_peaks) - k_ / n_) < 0.0015), "each log-likelihood should peak at k / n"
assert abs(k_[-1] / 640 - 0.8) < 0.08, "with 640 examples, k / n should land near p_true = 0.8"
print("ok  exercise 3 simulation")
check("p_perfect_10", p_perfect_10, "749e31eba588beb8")
check("p_perfect_40", p_perfect_40, "476cf4f14670a871")
print("ok  exercise 3 perfect scores")


**Write.** (a) Rerun the answer cell with two or three other seeds. Which peaks move a lot, and which barely move? How does the spread of the peaks compare with the widths of the curves? (b) A teammate's 10-example test comes back 10 of 10, and they report "accuracy 100%." Use `p_perfect_10` and the likelihood for 10 of 10, $L(p)=p^{10}$, to say what is wrong with that report. (c) Why is a likelihood curve not a probability distribution for $p$? Four to six sentences.

YOUR ANSWER HERE


## Exercise 4: why averages look Normal, and where they still do not

A support line's hold times are Exponential with mean 2 minutes. A single hold time has a very skewed distribution: most callers wait less than 2 minutes, and about 5% wait more than 6. For an Exponential the standard deviation equals the mean, so one hold time has $\sigma=2$.

The operations dashboard shows the average hold time $\bar X$ of the last $n$ callers and pages the manager when that average is above 2.5 minutes. Nothing is wrong with the line. How often does it page anyway?

1. **Theory.** The average of $n$ independent hold times has $\mathrm{sd}(\bar X)=\sigma/\sqrt n$, the same $1/\sqrt n$ that narrowed the likelihood curves in Exercise 3. Store `se_5`, `se_20`, `se_80`.
2. **The Normal's answer.** Treat $\bar X$ at $n=80$ as $N(2,\ \sigma^2/80)$. Store `p_page_normal` $=P(\bar X>2.5)$, the false-page rate that approximation predicts.
3. **Simulate.** `rng.exponential(scale=mu, size=(n_rep, n))` is a table with `n_rep` rows, each one sample of $n$ callers. `.mean(axis=1)` averages each row, giving `n_rep` values of $\bar X$. Do this for $n=5$ and $n=80$ (and $n=20$ if you like), and store the averages as `means_5` and `means_80`. Store `sd_means_80`, the standard deviation of `means_80`, and `p_page_sim`, the share of `means_80` above 2.5.
4. **Look.** `overlay_means(means, n)` draws a histogram of your averages with the Normal curve on top. Uncomment the two calls at the bottom of the starter.


**Predict before you simulate.** One line each.

1. From $n=20$ to $n=80$ callers, what happens to $\mathrm{sd}(\bar X)$?
2. What does the histogram of 10,000 averages look like at $n=5$? At $n=80$?
3. At $n=80$, is the Normal's false-page rate too high, too low, or about right?

YOUR PREDICTION HERE


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

mu, sigma = 2.0, 2.0     # one hold time: mean 2 minutes, sd 2 minutes
n_rep = 10000            # simulated samples at each n
rng = np.random.default_rng(80)


def overlay_means(means, n):
    se = sigma / np.sqrt(n)
    xs = np.linspace(mu - 4 * se, mu + 4 * se, 200)
    plt.figure()
    plt.hist(means, bins=40, density=True, color="steelblue", edgecolor="white")
    plt.plot(xs, norm.pdf(xs, loc=mu, scale=se), color="black", linewidth=2)
    if xs[0] < 0:
        plt.axvline(0, color="firebrick", linestyle="--")
    plt.title(f"n = {n}")
    plt.xlabel("average hold time (minutes)")
    plt.show()


se_5 = None
se_20 = None
se_80 = None
p_page_normal = None   # P(average of 80 > 2.5) under N(2, se_80**2)
means_5 = None         # n_rep simulated averages of 5 hold times
means_80 = None        # n_rep simulated averages of 80 hold times
sd_means_80 = None     # sd of means_80
p_page_sim = None      # share of means_80 above 2.5
# overlay_means(means_5, 5)
# overlay_means(means_80, 80)
se_5, se_20, se_80, p_page_normal, sd_means_80, p_page_sim


In [ ]:
check("se_5", se_5, "55dc5ef7b2d85d0a")
check("se_20", se_20, "77fdfdee2aba05ca")
check("se_80", se_80, "72fe62b29438e38e")
check("p_page_normal", p_page_normal, "9333dd8f527fcabe")
print("ok  exercise 4 theory")
assert n_rep >= 8000, "use at least 8,000 simulated samples"
assert sd_means_80 is not None and p_page_sim is not None, "replace every None"
assert abs(sd_means_80 - se_80) < 0.01, "the sd of means_80 should sit on se_80"
assert 0.009 < p_page_sim < 0.025, "p_page_sim is the share of means_80 above 2.5; check size and scale"
print("ok  exercise 4 simulation")


**Write.** (a) At $n=5$ the Normal curve runs past the red line at zero. What does it claim there that the histogram never does, and what does that say about using the Normal at $n=5$? (b) At $n=80$ the middle of the bell fits well. Compare `p_page_sim` with `p_page_normal`: does the tail fit, and why would the averages still have more upper tail than the Normal allows? (c) Which false-page rate do you give the manager, and why? Four to six sentences.

YOUR ANSWER HERE


## Exercise 5 (optional, hard): is 98.6°F normal?

Skip this if you want. Next week we maximise a Normal likelihood to fit a line; this is the same argument with no line, just a mean and a spread.

Everyone "knows" that normal body temperature is 98.6°F. The number traces to Carl Wunderlich, who published it in 1868. `bodytemp.txt` has 130 readings: Shoemaker (1996) rebuilt them from a histogram published by Mackowiak, Wasserman and Levine (1992), whose study took 700 readings from 148 adults over two days. Column `temperature`, in °F. The file is in the same folder as this notebook (or [the course copy](https://vsokolov.org/courses/files/600/bodytemp.txt)).

Model the readings as $X_1,\ldots,X_n$ iid $N(\mu,\sigma^2)$. Up to a constant,
$$\ell(\mu,\sigma^2)=-\frac n2\log\sigma^2-\frac1{2\sigma^2}\sum_{i=1}^n(x_i-\mu)^2.$$

1. On paper, set $\partial\ell/\partial\mu=0$ and solve. Store $\hat\mu$ as `mu_mle`. It is a one-line mean.
2. Plug $\hat\mu$ in, set $\partial\ell/\partial\sigma^2=0$, and solve. Store $\hat\sigma^2$ as `sigma2_mle`. The denominator comes out $n$, not $n-1$: `np.var(x)` divides by $n$, while `temp["temperature"].var()` divides by $n-1$.
3. How many standard errors does 98.6 sit above $\hat\mu$? Use $\mathrm{sd}(\bar X)=\sigma/\sqrt n$ from Exercise 4, with $\hat\sigma$ in place of $\sigma$: `z_986` $=(98.6-\hat\mu)\big/\sqrt{\hat\sigma^2/n}$.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

path = Path("bodytemp.txt")
if not path.exists():
    path = "https://vsokolov.org/courses/files/600/bodytemp.txt"
temp = pd.read_csv(path)
x = temp["temperature"].to_numpy()
n = len(x)

mu_mle = None
sigma2_mle = None
z_986 = None        # standard errors from mu_mle up to 98.6
mu_mle, sigma2_mle, z_986


In [ ]:
check("mu_mle", mu_mle, "e37ffcd0569ce5e6")
check("sigma2_mle", sigma2_mle, "443982cfbd47a481")
check("z_986", z_986, "0ffcb4817207d045")
print("ok  exercise 5 normal mle")


**Write.** (a) Is the gap between $\hat\mu$ and 98.6 larger than sampling noise? Then reread where the 130 rows came from. What in that story makes you doubt that $n=130$ independent people is the right $n$ for the standard error? Two or three sentences.

(b) For the Normal, maximum likelihood and the method of moments give the same $\hat\mu$ and $\hat\sigma^2$. They can disagree. Fit Uniform$(0,\theta)$ to the eight call times from Exercise 1 both ways. Moments: $E(X)=\theta/2$, so set $\theta/2=\bar x$. Likelihood: $L(\theta)=\theta^{-8}$ when $\theta$ is at least every observed time, and $0$ otherwise. Which estimate contradicts the data? Two sentences.

YOUR ANSWER HERE


## Before you submit

- Restart the kernel and run every cell. Each test cell for Exercises 1 to 4 prints `ok`. If you skipped Exercise 5, its test cell stops with `replace None`; that is fine.
- Every YOUR PREDICTION HERE and YOUR ANSWER HERE is replaced.
- Cheat sheet check. Without looking, can you write: $P(a<X\le b)$ from $F$; a Normal quantile through $z$; $F^{-1}$ for an Exponential with mean $\theta$; $\hat p$ for $k$ successes in $n$ trials; $\mathrm{sd}(\bar X)$, and what four times the data does to it? Whatever you cannot write goes on the sheet.
